# Tugas 03 : **Melakukan Pengujian dan Deploy**

NAMA : Mohammad Hasan Basri

NIM  : 210411100169

MATA KULIAH : Pencarian dan Penambangan Web - A

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pickle
import pandas as pd

Mounted at /content/drive


In [2]:
df = pd.read_csv("/content/drive/My Drive/PPWA/report/Tugas_PPWA/Hasil_Preprocessing_Hasan.csv")
df.head()

,judul,tanggal,isi,kategori,cleansing,case_folding,tokenize,stopword_removal
0,Iran Serukan Negara Muslim Bersatu Hentikan Ke...,"Kamis, 17 Okt 2024 10:01 WIB",Jakarta - Presiden Iran Masoud Pezeshkian mene...,Keislaman,Jakarta Presiden Iran Masoud Pezeshkian menek...,jakarta presiden iran masoud pezeshkian menek...,"['jakarta', 'presiden', 'iran', 'masoud', 'pez...",jakarta presiden iran masoud pezeshkian meneka...
1,"Bacaan Al-Qur'an Surah Asy-Syu'ara: Arab, Lati...","Kamis, 17 Okt 2024 09:30 WIB",NaN,Keislaman,NaN,NaN,['nan'],NaN
2,"Rukun Iman Ke-2, Setiap Muslim Wajib Mengimani...","Kamis, 17 Okt 2024 08:45 WIB",Jakarta - Malaikat adalah makhluk ciptaan Alla...,Keislaman,Jakarta Malaikat adalah makhluk ciptaan Allah...,jakarta malaikat adalah makhluk ciptaan allah...,"['jakarta', 'malaikat', 'adalah', 'makhluk', '...",jakarta malaikat makhluk ciptaan allah swt mus...
3,Kalender Ramadhan 2025 Muhammadiyah dan Predik...,"Kamis, 17 Okt 2024 08:00 WIB",Jakarta - Kalender Ramadhan 2025 versi Muhamma...,Keislaman,Jakarta Kalender Ramadhan versi Muhammadiyah...,jakarta kalender ramadhan versi muhammadiyah...,"['jakarta', 'kalender', 'ramadhan', 'versi', '...",jakarta kalender ramadhan versi muhammadiyah t...
4,20 Gambaran Kehidupan di Neraka Menurut Al-Qur...,"Kamis, 17 Okt 2024 07:15 WIB",Jakarta - Gambaran kehidupan di neraka selalu ...,Keislaman,Jakarta Gambaran kehidupan di neraka selalu m...,jakarta gambaran kehidupan di neraka selalu m...,"['jakarta', 'gambaran', 'kehidupan', 'di', 'ne...",jakarta gambaran kehidupan neraka pengingat ke...


**TF-IDF (Term Frequency-Inverse Document Frequency)**


---


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Menginisialisasi TfidfVectorizer
vectorizer = TfidfVectorizer()

# Mengisi nilai NaN pada kolom 'stopword_removal' dengan string kosong
df['stopword_removal'] = df['stopword_removal'].fillna('')

# Menghitung TF-IDF
tfidf_matrix = vectorizer.fit_transform(df['stopword_removal'])

# Mengubah hasilnya menjadi DataFrame
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df.head(10)
# Menyimpan hasil ke file CSV
tfidf_df.to_csv('hasil_tfidf.csv', index=False)

In [4]:
# Menggunakan kolom 'cleaned_text' sebagai teks input dan 'kategori' sebagai label
texts = df['stopword_removal'].fillna('')  # Mengganti NaN dengan string kosong jika ada
labels = df['kategori'].fillna('unknown')  # Mengganti NaN di kategori jika ada

# Melihat label yang digunakan untuk klasifikasi
print(labels.unique())

['Keislaman' 'Pariwisata']


**mengubah data teks menjadi representasi numerik menggunakan TF-IDF Vectorizer, kemudian membagi dataset yang sudah diubah ini menjadi data latih dan data uji menggunakan fungsi train_test_split.**

In [5]:
from sklearn.model_selection import train_test_split
# Melakukan transformasi TF-IDF
tfidf = TfidfVectorizer(max_features=None)  # Menggunakan 5000 fitur teratas
X_tfidf = tfidf.fit_transform(texts)

# Memisahkan dataset menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, labels, test_size=0.2, random_state=42)

# Melihat bentuk dari hasil TF-IDF
print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)

Shape of X_train: (80, 6772)
Shape of X_test: (20, 6772)


**membuat dan melatih model Logistic Regression dengan regularisasi untuk mencegah overfitting, kemudian menggunakan model tersebut untuk memprediksi hasil pada data uji.**

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Membuat model Logistic Regression dengan regularisasi
model = LogisticRegression(max_iter=1000, C=0.3)  # Mengurangi nilai C untuk menambah regularisasi

# Melatih model dengan data yang sudah diresample (jika diperlukan)
model.fit(X_train, y_train)

# Memprediksi hasil pada data testing
y_pred = model.predict(X_test)

**Evaluasi Hasil**

---


In [7]:
# Evaluasi hasil prediksi
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

# Menampilkan laporan klasifikasi
print(classification_report(y_test, y_pred))

Accuracy: 75.00%
              precision    recall  f1-score   support

   Keislaman       1.00      0.58      0.74        12
  Pariwisata       0.62      1.00      0.76         8

    accuracy                           0.75        20
   macro avg       0.81      0.79      0.75        20
weighted avg       0.85      0.75      0.75        20



**Menyimpan model dan vectorizer (TF-IDF dan Logistic Regression) ke dalam file menggunakan pickle**

---


In [8]:
# Menyimpan tfidf_matrix dan vectorizer menggunakan pickle
with open('tfidf_vectorizer.pkl', 'wb') as file:
    pickle.dump(vectorizer, file)
# Menyimpan model ke dalam file pickle
with open('logistic_regression_model.pkl', 'wb') as file:
    pickle.dump(model, file)

**PENGUJIAN**

---



In [9]:
import joblib

# Memuat vectorizer dan model
tfidf = joblib.load('tfidf_vectorizer.pkl')
model = joblib.load('logistic_regression_model.pkl')

In [10]:
# Teks baru yang ingin diprediksi
new_text = ['''Jakarta  Malaikat adalah makhluk ciptaan Allah SWT Setiap muslim wajib mengimani adanya malaikat hal ini sebagaimana tercantum sebagai rukun iman ke Iman kepada malaikat menjadi kewajiban bagi setiap muslim Siapapun yang bertakwa kepada Allah SWT maka harus meyakini adanya malaikat sebagai makhluk ciptaan Allah SWT Mengutip buku berjudul Rukun Iman karya Hudarrohmah dijelaskan pengertian dari rukun iman Rukun artinya dasar atau pokok yang harus dikerjakan Sementara iman artinya yakin atau percaya Dengan demikian rukun iman merupakan dasar atau pokok kepercayaan Ciri Muslim Beriman Rukun iman dituangkan dalam diri manusia yang beriman meliputi tiga tahap yakni  Iman diyakini dalam hati  Iman diikrarkan dengan lisan  Iman diamalkan dengan anggota badan Tiga hal ini menjadi penentu keimanan seseorang Iman diyakini dalam hati yaitu mempercayai dan meyakini dengan sepenuh hati bahwa adanya alam semesta dan segala isinya itu pasti ada yang menciptakan dan ada yang mengatur yakni Allah SWT Kemudian iman yang diikrarkan dengan lisan yakni mengucapkan dengan sungguhsungguh bahwa ia beriman kepada Allah SWT kepada malaikatmalaikat kitabkitab rasulrasul hari akhir dan juga meyakini keimanan pada ketetapan baik dan buruk Dan tanda seorang muslim yang beriman selanjutnya adalah keimanan dikerjakan dengan anggota badan yaitu dengan menjalani segala perintah Allah SWT dan menjauhi segala larangan Allah SWT Bila seseorang mengaku beriman maka ia akan menunjukkannya dengan sikap takwa Karena keimanan akan sempurna apabila diiringi dengan ketakwaan Rukun Iman Ke Dalam hadits Rasulullah SAW bersabda Keimanan itu ialah engkau akan percaya beriman pada Allah malaikatmalaikatNya kitabkitabNya rasulrasulNya hari akhir kiamat dan engkau anak percaya kepada takdir baik dan buruk padaNya HR Muslim Merujuk pada hadits tersebut maka rukun iman terdiri dari enam berikut rinciannya  Iman kepada Allah SWT  Iman kepada malaikat  Iman kepada kitab  Iman kepada rasul  Iman kepada hari akhir  Iman kepada qada dan qadar Malaikat diciptakan dari cahaya sebagaimana sabda Rasulullah SAW Malaikat itu diciptakan dari cahaya Jin diciptakan dari api yang menyalanyala sedangkan Adam diciptakan dari apa yang telah dijelaskan kepada kalian HR Muslim Quraish Shihab dalam bukunya yang berjudul Malaikat dalam AlQuran Yang Halus dan Tak Terlihat menyebutkan setidaknya ada  malaikat beserta tugasnya yang wajib diimani oleh umat muslim Meskipun sebenarnya jumlah malaikat sangat banyak dan tidak terhitung jumlahnya Setiap malaikat memiliki tugasnya masingmasing sesuai dengan perintah Allah SWT Meskipun tidak tampak manusia wajib meyakini keberadaan malaikat Mengutip buku Islamologi Malaikat karya Maulana Muhammad Ali salah satu tugas malaikat yakni mencatat perbuatan manusia yang baik dan yang buruk Malaikat dengan tugas ini disebut kiraman katibin yang artinya juru tulis yang mulia Dalil dan Tugas Malaikat Ada beberapa ayat AlQuran yang menjelaskan nama malaikat lengkap dengan tugasnya  Surat AnNahl Ayat               Artinya Katakanlah Ruhul Qudus Jibril menurunkan Al Quran itu dari Tuhanmu dengan benar untuk meneguhkan hati orangorang yang telah beriman dan menjadi petunjuk serta kabar gembira bagi orangorang yang berserah diri kepada Allah  Surat AlBaqarah Ayat              Artinya Barang siapa yang menjadi musuh Allah malaikatmalaikatNya rasulrasulNya Jibril dan Mikail maka sesungguhnya Allah adalah musuh orangorang kafir  Surat AzZumar Ayat                         Artinya Dan ditiuplah sangkakala maka matilah siapa yang di langit dan di bumi kecuali siapa yang dikehendaki Allah Kemudian ditiup sangkakala itu sekali lagi maka tibatiba mereka berdiri menunggu putusannya masingmasing  Surat AsSajdah Ayat              Artinya Katakanlah Malaikat maut yang diserahi untuk mencabut nyawamu akan mematikanmu kemudian hanya kepada Tuhanmulah kamu akan dikembalikan  Surat Qaf Ayat          Artinya yaitu ketika dua orang malaikat mencatat amal perbuatannya seorang duduk di sebelah kanan dan yang lain duduk di sebelah kiri dvslus''']

# Preprocessing dan transformasi menggunakan TF-IDF yang sudah disimpan
new_text_tfidf = tfidf.transform(new_text)

# Melakukan prediksi menggunakan model Logistic Regression
prediction = model.predict(new_text_tfidf)

# Hasil prediksi
predicted_category = "Keislaman" if prediction[0] == 'Keislaman' else "Pariwisata"
print(f"Prediksi: {predicted_category}")

Prediksi: Keislaman


In [11]:
# Mengecek versi scikit-learn
import sklearn
print(sklearn.__version__)

1.6.0
